# Compositional analysis: preclinical snRNA-seq (pig-to-baboon cardiac xHTx)

For every heart, we count the nuclei of each cell type and cell state, turn the
counts into percentages, and draw them per condition. The P-value, FDR, effect size, fold change and log2FC are read from the CLR analysis and fetched here.


## 1. Imports and plotting style

In [1]:
# =============================================================================
# 1. IMPORTS AND PLOTTING STYLE
# =============================================================================
import glob
import os

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

%matplotlib inline

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=300, dpi_save=300)

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

sns.set_style("white")

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

## 2. Configuration


In [2]:
# =============================================================================
# 2. CONFIGURATION
# =============================================================================

OBJECT_PATH = (
    "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/"
    "global_object_assembly/objects/"
    "global_X_UMAP_Added_PCMV_score_updated_celltype_cellstate_annotated_preclinical_object_RAW_v3.h5ad"
)
CLR_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/CLR/v1"
CLR_CELL_TYPE_FILES = {          # reference arm -> (per-analysis Stats file, WholeMaster file used as a cross-check)
    "3month-protocol": (f"{CLR_DIR}/cell_type/ref_3month/results/CLR_CellType_Ref_3month_Stats.csv",
                        f"{CLR_DIR}/cell_type/ref_3month/results/CLR_CellType_Ref_3month_WholeMaster.csv"),
    "Control":         (f"{CLR_DIR}/cell_type/ref_control/results/CLR_CellType_Ref_Control_Stats.csv",
                        f"{CLR_DIR}/cell_type/ref_control/results/CLR_CellType_Ref_Control_WholeMaster.csv"),
}
CLR_CELL_STATE_REF = "3month-protocol"
CLR_CELL_STATE_FILE = f"{CLR_DIR}/cell_state/ref_3month/{{ct}}/results/CLR_CellState_{{ct}}_Stats.csv"
CLR_CELL_STATE_MASTER = f"{CLR_DIR}/cell_state/ref_3month/CLR_CellState_WholeMaster.csv"

# Pseudocount of the CLR cell-state script (used only to re-derive CLR's fold change as a check in step 7)
CLR_PSEUDOCOUNT_CELL_STATE = 0.2

# --- Output folder ---
OUT_DIR = os.environ.get(
    "XENO_COMPOSITION_OUT",
    "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/composition",
)
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FILE_TAG = "preclinical"

# --- Columns of .obs ---
SAMPLE_COL = "Sample_ID"       # one heart = the unit
CONDITION_COL = "Condition"
CELLTYPE_COL = "cell_type"
CELLSTATE_COL = "cell_state"

# --- Study arms: x-axis order (left to right) and the project colours ---
CONDITION_ORDER = [
    "Control",
    "3month-protocol",
    "6month-protocol",
    "Prematurely-terminated",
    "PCMV/PRV",
    "Rejection",
]
COND_COLORS = {
    "Control": "#EFE48E",
    "3month-protocol": "skyblue",
    "6month-protocol": "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV": "#F0C096",
    "Rejection": "#E09E9E",
}
REF_TAG = {"3month-protocol": "ref_3month", "Control": "ref_control"}

# --- Significance marks on the CLR FDR ---
STAR_THRESHOLDS = [(0.001, "***"), (0.01, "**"), (0.05, "*")]
STAR_MARKS = [mark for _, mark in STAR_THRESHOLDS]

# --- Figure sets: output folder -> (level, reference arm, arms shown left to right) ---
FIGURE_SETS = {
    "cell_type_all/ref_3month":   ("cell_type",  "3month-protocol", CONDITION_ORDER),
    "cell_type_all/ref_control":  ("cell_type",  "Control",         CONDITION_ORDER),
    "cell_state_all":             ("cell_state", "3month-protocol", CONDITION_ORDER),
    "cell_state_6m_3m":           ("cell_state", "3month-protocol", ["3month-protocol", "6month-protocol"]),
    "cell_state_3month_PCMV_Rej": ("cell_state", "3month-protocol", ["3month-protocol", "PCMV/PRV", "Rejection"]),
    "cell_state_3month_Rej":      ("cell_state", "3month-protocol", ["3month-protocol", "Rejection"]),
}

# --- Figure geometry ---
FIGSIZE_BY_NCOND = {        # width, height of ONE subplot, by number of arms shown
    2: (1.5, 3.5),
    3: (3.0, 3.0),
    4: (4.0, 4.0),
    5: (4.0, 5.0),
    6: (6.0, 6.0),
}
N_PANEL_COLS = 4
BAR_WIDTH, BAR_SPACING, INNER_XPAD = 0.5, 0.6, 0.3
POINT_JITTER, POINT_SIZE, POINT_ALPHA = 0.06, 30, 0.70
ERRORBAR_LW, ERRORBAR_CAPSIZE = 1.5, 4
Y_START_ABOVE_DATA, Y_STEP, BRACKET_H_FRAC, STAR_OFFSET_FRAC = 0.06, 0.09, 0.02, 0.01
JITTER_SEED = 12345         # fixes the horizontal jitter of the dots

print(f"Output folder : {OUT_DIR}")
print(f"CLR folder    : {CLR_DIR}")

Output folder : /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/composition
CLR folder    : /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/CLR/v1


## 3. Load the cell metadata and check the design

In [3]:
# =============================================================================
# 3. LOAD THE CELL METADATA 
# =============================================================================
with h5py.File(OBJECT_PATH, "r") as f:
    obs_all = ad.io.read_elem(f["obs"])

obs = obs_all[[SAMPLE_COL, CONDITION_COL, CELLTYPE_COL, CELLSTATE_COL]].copy()
del obs_all

missing = obs.isna().sum()
assert missing.sum() == 0, f"Missing labels:\n{missing[missing > 0]}"

arms_per_heart = obs.groupby(SAMPLE_COL, observed=True)[CONDITION_COL].nunique()
assert (arms_per_heart == 1).all(), f"Hearts in >1 arm: {list(arms_per_heart[arms_per_heart > 1].index)}"

absent_arms = set(CONDITION_ORDER) - set(obs[CONDITION_COL].astype(str))
assert not absent_arms, f"Configured arms absent from the object: {absent_arms}"

# Hearts in the object's category order (this order also fixes the dot jitter of the figures)
present_samples = set(obs[SAMPLE_COL].astype(str))
SAMPLE_ORDER = [s for s in obs[SAMPLE_COL].cat.categories if s in present_samples]
SAMPLE_CONDITION = (
    obs.groupby(SAMPLE_COL, observed=True)[CONDITION_COL].first().astype(str).reindex(SAMPLE_ORDER)
)

# Cell types in category order; cell states in category order within each parent cell type
present_types = set(obs[CELLTYPE_COL].astype(str))
CELL_TYPE_ORDER = [c for c in obs[CELLTYPE_COL].cat.categories if c in present_types]
states_by_type = obs.groupby(CELLTYPE_COL, observed=True)[CELLSTATE_COL].unique()
CELL_STATE_ORDER = {
    ct: [s for s in obs[CELLSTATE_COL].cat.categories if s in set(states_by_type[ct])]
    for ct in CELL_TYPE_ORDER
}
# Cell types with a single state (EC_papio, MC, NC) have no state composition to analyse
CELL_STATE_PARENTS = [ct for ct in sorted(CELL_TYPE_ORDER) if len(CELL_STATE_ORDER[ct]) > 1]

design = (
    pd.DataFrame({"condition": SAMPLE_CONDITION,
                  "nuclei": obs[SAMPLE_COL].astype(str).value_counts().reindex(SAMPLE_ORDER)})
    .groupby("condition")["nuclei"]
    .agg(hearts="size", nuclei="sum", nuclei_min="min", nuclei_max="max")
    .reindex(CONDITION_ORDER)
)
print(f"{len(obs):,} nuclei from {len(SAMPLE_ORDER)} hearts\n")
print(design.to_string(), "\n")
print("Cell states per cell type:")
for ct in CELL_TYPE_ORDER:
    print(f"  {ct:<12} {len(CELL_STATE_ORDER[ct]):>2}  {CELL_STATE_ORDER[ct]}")
print(f"\nCell-state analysis for: {CELL_STATE_PARENTS}")

233,200 nuclei from 26 hearts

                        hearts  nuclei  nuclei_min  nuclei_max
condition                                                     
Control                      4   29202        6494        8221
3month-protocol              7   72986        6678       13409
6month-protocol              4   31989        6046       10393
Prematurely-terminated       3   29136        8288       11910
PCMV/PRV                     3   20374        4873        8294
Rejection                    5   49513        6807       14915 

Cell states per cell type:
  CM            7  ['CM1', 'CM2', 'CM3', 'CM4', 'CM5', 'Hypoxia_CM', 'ISG_CM']
  FB            8  ['Angio_FB', 'ECM_FB', 'Homeo_FB', 'Hypoxia_FB', 'ISG_FB', 'Myo_FB', 'Progenitor_FB', 'Prolif_Myo_FB']
  EC            9  ['Arterial_EC', 'Cap_EC', 'Hypoxia_EC', 'ISG_cap_EC', 'Lymph_EC', 'Meso_EC', 'PCMV/PRV_EC', 'Prolif_EC', 'Ven_EC']
  EC_papio      1  ['EC_papio']
  MC            1  ['MC']
  Mye_papio     9  ['AP_MP_papio', 'DC_papi

## 4. Composition functions
Counts → per-heart percentages → one summary row per population and arm. The same functions serve both levels.

In [4]:
# =============================================================================
# 4. COMPOSITION FUNCTION
# =============================================================================

def per_sample_composition(cells, unit_col, unit_order, parent_label=None):
    """
    One row per population (cell type or cell state) x heart.

    abundance         nuclei of the population in the heart
    abundance_parent  nuclei of the reference population in the heart
                      (all nuclei for cell types; the parent cell type for cell states)
    proportion        abundance / abundance_parent * 100; blank if abundance_parent = 0

    Every heart of the cohort is listed and missing populations are written as 0,
    so a heart without a state contributes 0 %, not a missing value.
    """
    counts = pd.crosstab(cells[SAMPLE_COL].astype(str), cells[unit_col].astype(str))
    counts = counts.reindex(index=SAMPLE_ORDER, columns=unit_order, fill_value=0)
    counts.index.name, counts.columns.name = SAMPLE_COL, unit_col

    long = counts.reset_index().melt(id_vars=SAMPLE_COL, var_name=unit_col, value_name="abundance")
    long["condition"] = long[SAMPLE_COL].map(SAMPLE_CONDITION)
    long["abundance_parent"] = long[SAMPLE_COL].map(counts.sum(axis=1))
    with np.errstate(invalid="ignore", divide="ignore"):
        long["proportion"] = (long["abundance"] / long["abundance_parent"]) * 100
    long.loc[long["abundance_parent"] == 0, "proportion"] = np.nan

    # rows: population, then arm, then heart (object order) -- the order the dots are drawn in
    long["_u"] = long[unit_col].map({u: i for i, u in enumerate(unit_order)})
    long["_c"] = long["condition"].map({c: i for i, c in enumerate(CONDITION_ORDER)})
    long = long.sort_values(["_u", "_c"], kind="mergesort").reset_index(drop=True)

    lead = [unit_col]
    if parent_label is not None:
        long.insert(0, CELLTYPE_COL, parent_label)
        lead = [CELLTYPE_COL, unit_col]
    return long[lead + ["condition", SAMPLE_COL, "abundance", "abundance_parent", "proportion"]]


def summarise_by_condition(per_sample, unit_col, unit_order, parent_label=None):
    """
    One row per population x arm.

    n_samples          hearts with >= 1 nucleus of the reference population (the dots of the figure)
    abundance          nuclei of the population, summed over the arm's hearts
    abundance_parent   nuclei of the reference population, summed over the arm's hearts
    proportion         mean of the per-heart percentages  (bar height)
    proportion_sem     SEM of the per-heart percentages   (error bar; blank if n_samples < 2)
    proportion_median  median of the per-heart percentages
    """
    rows = []
    for unit in unit_order:
        d_unit = per_sample[per_sample[unit_col] == unit]
        for cond in CONDITION_ORDER:
            d = d_unit[d_unit["condition"] == cond]
            vals = d["proportion"].dropna().to_numpy()
            n = len(vals)
            n_nuc, n_par = int(d["abundance"].sum()), int(d["abundance_parent"].sum())
            row = {} if parent_label is None else {CELLTYPE_COL: parent_label}
            row.update({
                unit_col: unit,
                "condition": cond,
                "n_samples": n,
                "abundance": n_nuc,
                "abundance_parent": n_par,
                "proportion": np.mean(vals) if n > 0 else np.nan,
                "proportion_sem": np.std(vals, ddof=1) / np.sqrt(n) if n > 1 else np.nan,
                "proportion_median": np.median(vals) if n > 0 else np.nan,
            })
            rows.append(row)
    return pd.DataFrame(rows)

## 5. Composition tables

In [5]:
# =============================================================================
# 5. COMPOSITION TABLES (both levels)
# In  : obs
# Out : ct_per_sample, ct_per_condition   (cell types, % of all nuclei of the heart)
#       cs_per_sample, cs_per_condition   (cell states, % of the parent cell type; all parents stacked)
# =============================================================================
ct_per_sample = per_sample_composition(obs, CELLTYPE_COL, CELL_TYPE_ORDER)
ct_per_condition = summarise_by_condition(ct_per_sample, CELLTYPE_COL, CELL_TYPE_ORDER)

ps_list, pc_list = [], []
for ct in CELL_STATE_PARENTS:
    cells = obs[obs[CELLTYPE_COL] == ct]
    ps = per_sample_composition(cells, CELLSTATE_COL, CELL_STATE_ORDER[ct], parent_label=ct)
    ps_list.append(ps)
    pc_list.append(summarise_by_condition(ps, CELLSTATE_COL, CELL_STATE_ORDER[ct], parent_label=ct))
cs_per_sample = pd.concat(ps_list, ignore_index=True)
cs_per_condition = pd.concat(pc_list, ignore_index=True)

print(f"cell type : {len(ct_per_sample)} heart rows, {len(ct_per_condition)} arm rows")
print(f"cell state: {len(cs_per_sample)} heart rows, {len(cs_per_condition)} arm rows")

cell type : 260 heart rows, 60 arm rows
cell state: 1430 heart rows, 330 arm rows


## 6. Load the CLR results

In [6]:
# =============================================================================
# 6. LOAD THE CLR RESULTS
# read the CLR statistics for cell types (reference 3month-protocol and Control)
# and cell states (reference 3month-protocol, one file per parent cell type)
# =============================================================================
CLR_VALUE_COLS = ["EffectSize", "MeanPropRef", "fold_change", "log2FC", "PValue", "FDR"]


def read_clr(stats_path, ref, unit_col, parent_label=None):
    clr = pd.read_csv(stats_path).rename(columns={
        "Condition": "comp_condition", "Target": unit_col, "MeanPropControl": "MeanPropRef"})
    clr.insert(0, "ref_condition", ref)
    if parent_label is not None:
        clr.insert(0, CELLTYPE_COL, parent_label)
    key = ([CELLTYPE_COL] if parent_label is not None else []) + [unit_col, "comp_condition"]
    assert not clr.duplicated(key).any(), f"duplicated rows in {stats_path}"
    assert set(clr["comp_condition"]) <= set(CONDITION_ORDER) - {ref}, f"unexpected arms in {stats_path}"
    return clr[key[:-1] + ["ref_condition", "comp_condition"] + CLR_VALUE_COLS]


def assert_same_as_master(clr, master_path, unit_col, ref):
    """The per-analysis Stats file and the CLR WholeMaster file must hold the same numbers."""
    master = pd.read_csv(master_path).rename(columns={"Condition": "comp_condition",
                                                      "Ref_condition": "ref_condition",
                                                      "MeanPropControl": "MeanPropRef"})
    master = master[master["ref_condition"] == ref]
    assert len(master) == len(clr), f"{master_path}: {len(master)} rows, Stats files {len(clr)} rows"
    key = [c for c in [CELLTYPE_COL, CELLSTATE_COL] if c in clr.columns and c in master.columns]
    m = clr.merge(master, on=key + ["ref_condition", "comp_condition"], suffixes=("", "_m"),
                  validate="one_to_one")
    assert len(m) == len(clr), f"{master_path}: rows differ from the Stats files"
    for c in CLR_VALUE_COLS:
        assert np.allclose(m[c], m[f"{c}_m"], rtol=1e-12, equal_nan=True), (master_path, c)


# --- cell types: one file per reference ---
clr_ct_list = []
for ref, (stats_path, master_path) in CLR_CELL_TYPE_FILES.items():
    clr = read_clr(stats_path, ref, CELLTYPE_COL)
    assert_same_as_master(clr, master_path, CELLTYPE_COL, ref)
    clr_ct_list.append(clr)
    print(f"cell type, ref {ref:<16}: {len(clr):>3} CLR rows  <- {os.path.relpath(stats_path, CLR_DIR)}")
clr_cell_type = pd.concat(clr_ct_list, ignore_index=True)

# --- cell states: one file per parent cell type (reference 3month-protocol) ---
clr_cs_list = []
for ct in CELL_STATE_PARENTS:
    path = CLR_CELL_STATE_FILE.format(ct=ct)
    if not os.path.exists(path):
        print(f"cell state, {ct:<12}: no CLR result file (CLR produced no test for this cell type)")
        continue
    clr = read_clr(path, CLR_CELL_STATE_REF, CELLSTATE_COL, parent_label=ct)
    clr_cs_list.append(clr)
    print(f"cell state, {ct:<12}: {len(clr):>3} CLR rows  <- {os.path.relpath(path, CLR_DIR)}")
clr_cell_state = pd.concat(clr_cs_list, ignore_index=True)
assert_same_as_master(clr_cell_state, CLR_CELL_STATE_MASTER, CELLSTATE_COL, CLR_CELL_STATE_REF)
print("\nEvery Stats file equals its CLR WholeMaster file: OK")

cell type, ref 3month-protocol :  50 CLR rows  <- cell_type/ref_3month/results/CLR_CellType_Ref_3month_Stats.csv
cell type, ref Control         :  50 CLR rows  <- cell_type/ref_control/results/CLR_CellType_Ref_Control_Stats.csv
cell state, CM          :  35 CLR rows  <- cell_state/ref_3month/CM/results/CLR_CellState_CM_Stats.csv
cell state, EC          :  45 CLR rows  <- cell_state/ref_3month/EC/results/CLR_CellState_EC_Stats.csv
cell state, FB          :  40 CLR rows  <- cell_state/ref_3month/FB/results/CLR_CellState_FB_Stats.csv
cell state, Lymph_papio :  40 CLR rows  <- cell_state/ref_3month/Lymph_papio/results/CLR_CellState_Lymph_papio_Stats.csv
cell state, Lymph_sus   : no CLR result file (CLR produced no test for this cell type)
cell state, Mye_papio   :  36 CLR rows  <- cell_state/ref_3month/Mye_papio/results/CLR_CellState_Mye_papio_Stats.csv
cell state, Mye_sus     :  50 CLR rows  <- cell_state/ref_3month/Mye_sus/results/CLR_CellState_Mye_sus_Stats.csv

Every Stats file equals 

## 7. Statistics tables: composition + CLR

In [7]:
# =============================================================================
# 7. STATISTICS TABLES = composition of both arms + CLR statistics
# =============================================================================

def significance_stars(fdr):
    """'***' / '**' / '*' for FDR < 0.001 / 0.01 / 0.05; 'ns' if tested and not significant; '' if no CLR result."""
    if pd.isna(fdr):
        return ""
    for threshold, mark in STAR_THRESHOLDS:
        if fdr < threshold:
            return mark
    return "ns"


def build_stats(per_condition, clr, unit_col, unit_order, ref, parent_label=None):
    summary = per_condition.set_index([unit_col, "condition"])
    rows = []
    for comp in [c for c in CONDITION_ORDER if c != ref]:
        for unit in unit_order:
            s_ref, s_comp = summary.loc[(unit, ref)], summary.loc[(unit, comp)]
            row = {} if parent_label is None else {CELLTYPE_COL: parent_label}
            row.update({
                unit_col: unit,
                "ref_condition": ref,
                "comp_condition": comp,
                "n_samples_ref": int(s_ref["n_samples"]),
                "n_samples_comp": int(s_comp["n_samples"]),
                "abundance_in_ref": int(s_ref["abundance"]),
                "abundance_in_comp": int(s_comp["abundance"]),
                "proportion_in_ref": s_ref["proportion"],
                "proportion_in_comp": s_comp["proportion"],
            })
            rows.append(row)
    stats = pd.DataFrame(rows)
    key = ([CELLTYPE_COL] if parent_label is not None else []) + [unit_col, "ref_condition", "comp_condition"]
    clr_part = clr[(clr["ref_condition"] == ref)
                   & ((clr[CELLTYPE_COL] == parent_label) if parent_label is not None else True)]
    stats = stats.merge(clr_part, on=key, how="left", validate="one_to_one", indicator=True)
    unmatched = clr_part.merge(stats[key], on=key, how="left", indicator="_in").query("_in == 'left_only'")
    assert unmatched.empty, f"CLR rows without a composition row:\n{unmatched[key]}"
    assert (stats["_merge"] == "both").sum() == len(clr_part)
    stats = stats.drop(columns="_merge")
    stats["significance"] = stats["FDR"].map(significance_stars)
    return stats


def sort_stats(stats, unit_col, unit_order):
    """Comparisons in arm order; within each, most significant first (ties keep the panel order)."""
    out = stats.copy()
    out["_c"] = out["comp_condition"].map({c: i for i, c in enumerate(CONDITION_ORDER)})
    out["_u"] = out[unit_col].map({u: i for i, u in enumerate(unit_order)})
    out = out.sort_values(["_c", "FDR", "_u"], na_position="last", kind="mergesort")
    return out.drop(columns=["_c", "_u"]).reset_index(drop=True)


# --- cell types: one table per reference, stacked ---
ct_stats = pd.concat(
    [sort_stats(build_stats(ct_per_condition, clr_cell_type, CELLTYPE_COL, CELL_TYPE_ORDER, ref),
                CELLTYPE_COL, CELL_TYPE_ORDER)
     for ref in CLR_CELL_TYPE_FILES], ignore_index=True)

# --- cell states: one table per parent cell type, stacked ---
cs_stats = pd.concat(
    [sort_stats(build_stats(cs_per_condition, clr_cell_state, CELLSTATE_COL, CELL_STATE_ORDER[ct],
                            CLR_CELL_STATE_REF, parent_label=ct),
                CELLSTATE_COL, CELL_STATE_ORDER[ct])
     for ct in CELL_STATE_PARENTS], ignore_index=True)

assert ct_stats["PValue"].notna().sum() == len(clr_cell_type)
assert cs_stats["PValue"].notna().sum() == len(clr_cell_state)

# --- check: re-derive CLR's cell-state fold_change from the counts (CLR: +0.2 per state, then closure) ---
for ct in CELL_STATE_PARENTS:
    d = cs_per_sample[(cs_per_sample[CELLTYPE_COL] == ct) & (cs_per_sample["abundance_parent"] > 0)].copy()
    n_states = len(CELL_STATE_ORDER[ct])
    d["p_clr"] = (d["abundance"] + CLR_PSEUDOCOUNT_CELL_STATE) / (
        d["abundance_parent"] + CLR_PSEUDOCOUNT_CELL_STATE * n_states)
    mean_p = d.groupby([CELLSTATE_COL, "condition"])["p_clr"].mean()
    s = cs_stats[(cs_stats[CELLTYPE_COL] == ct) & cs_stats["PValue"].notna()]
    for _, r in s.iterrows():
        u, ref, comp = r[CELLSTATE_COL], r["ref_condition"], r["comp_condition"]
        assert np.isclose(r["fold_change"], mean_p[(u, comp)] / mean_p[(u, ref)], rtol=1e-9), (ct, u, comp)
print("CLR fold_change re-derived from the counts for every cell-state row: OK\n")

for name, st in [("cell type", ct_stats), ("cell state", cs_stats)]:
    print(f"{name:<10}: {len(st)} rows, {st['PValue'].notna().sum()} with a CLR result, "
          f"{st['significance'].isin(STAR_MARKS).sum()} with FDR < 0.05")
no_clr = cs_stats[cs_stats["PValue"].isna()].groupby([CELLTYPE_COL, "comp_condition"]).size()
print("\nCell-state comparisons without a CLR result (rows):")
print(no_clr.to_string())

CLR fold_change re-derived from the counts for every cell-state row: OK

cell type : 100 rows, 100 with a CLR result, 35 with FDR < 0.05
cell state: 275 rows, 246 with a CLR result, 54 with FDR < 0.05

Cell-state comparisons without a CLR result (rows):
cell_type    comp_condition        
Lymph_papio  Control                   10
Lymph_sus    6month-protocol            2
             Control                    2
             PCMV/PRV                   2
             Prematurely-terminated     2
             Rejection                  2
Mye_papio    Control                    9


In [8]:
# =============================================================================
# 8. SAVING CSVs
# =============================================================================

def save_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False, float_format="%.6g")
    print(f"  saved {df.shape[0]:>4} rows -> {os.path.relpath(path, OUT_DIR)}")


for level, tables in [("cell_type", [("per_sample", ct_per_sample), ("per_condition", ct_per_condition),
                                     ("stats", ct_stats)]),
                      ("cell_state", [("per_sample", cs_per_sample), ("per_condition", cs_per_condition),
                                      ("stats", cs_stats)])]:
    for kind, df in tables:
        save_csv(df, os.path.join(TABLE_DIR, f"{level}_composition_{kind}_{FILE_TAG}.csv"))

  saved  260 rows -> tables/cell_type_composition_per_sample_preclinical.csv
  saved   60 rows -> tables/cell_type_composition_per_condition_preclinical.csv
  saved  100 rows -> tables/cell_type_composition_stats_preclinical.csv
  saved 1430 rows -> tables/cell_state_composition_per_sample_preclinical.csv
  saved  330 rows -> tables/cell_state_composition_per_condition_preclinical.csv
  saved  275 rows -> tables/cell_state_composition_stats_preclinical.csv


## 9. Saving CSVs: the tables behind each figure


In [2]:
# =============================================================================
# 9. SAVE CSVs ; PER FIGURE SET
# =============================================================================

def figure_tables(level, ref, conditions, parent=None):
    """Composition and statistics of one figure: the arms shown and the comparisons between them."""
    if level == "cell_type":
        ps, pc, st = ct_per_sample, ct_per_condition, ct_stats
    else:
        ps, pc, st = (df[df[CELLTYPE_COL] == parent] for df in (cs_per_sample, cs_per_condition, cs_stats))
    st = st[(st["ref_condition"] == ref) & st["comp_condition"].isin(conditions)]
    assert ref in conditions
    return (ps[ps["condition"].isin(conditions)], pc[pc["condition"].isin(conditions)], st)


def figure_paths(folder, level, ref, parent=None):
    """Base path (without extension) of a figure and of its two CSVs."""
    if level == "cell_type":
        base = os.path.join(OUT_DIR, folder, "global_cell_types")
        return {"figure": f"{base}_composition_{REF_TAG[ref]}_{FILE_TAG}",
                "stats": f"{base}_composition_{REF_TAG[ref]}_{FILE_TAG}.csv",
                "per_condition": None}
    safe = parent.replace("/", "_").replace(" ", "_")
    base = os.path.join(OUT_DIR, folder, safe, f"{safe}_states")
    return {"figure": f"{base}_composition_{FILE_TAG}",
            "stats": f"{base}_composition_{FILE_TAG}.csv",
            "per_condition": f"{base}_abundance_per_condition_{FILE_TAG}.csv"}


save_csv(ct_per_condition,
         os.path.join(OUT_DIR, "cell_type_all", f"global_cell_types_abundance_per_condition_{FILE_TAG}.csv"))
for folder, (level, ref, conditions) in FIGURE_SETS.items():
    print(f"\n=== {folder}")
    for parent in ([None] if level == "cell_type" else CELL_STATE_PARENTS):
        ps, pc, st = figure_tables(level, ref, conditions, parent)
        paths = figure_paths(folder, level, ref, parent)
        save_csv(st, paths["stats"])
        if paths["per_condition"] is not None:
            save_csv(pc, paths["per_condition"])

## 10. Save comprehensive excel files

In [10]:
# =============================================================================
# 10. Save excel files
# 1 workbook per level with the pages Guide, Per condition, Per sample, Statistics.
# =============================================================================
from datetime import date
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

TITLE_FONT = Font(bold=True, size=14)
SECTION_FONT = Font(bold=True, size=12)
NOTE_FONT = Font(italic=True, size=10, color="404040")
HEADER_FONT = Font(bold=True, color="FFFFFF")
HEADER_FILL = PatternFill("solid", fgColor="2F5597")
HEADER_BORDER = Border(bottom=Side(style="thin", color="000000"))
NUMBER_FORMATS = {      # display only; the stored value keeps full precision
    "proportion": "0.000", "proportion_sem": "0.000", "proportion_median": "0.000",
    "proportion_in_ref": "0.000", "proportion_in_comp": "0.000",
    "EffectSize": "0.000", "log2FC": "0.000", "PValue": "0.00E+00", "FDR": "0.00E+00",
}
DATA_PAGES = ["Per condition", "Per sample", "Statistics"]


def level_words(level):
    if level == "cell_type":
        return {"unit": "cell type", "units": "cell types", "parent": "all nuclei of the heart", "parent_noun": "all nuclei",
                "ref": "3month-protocol and Control (see ref_condition)", "family": "all cell types"}
    return {"unit": "cell state", "units": "cell states", "parent": "the parent cell type's nuclei in that heart",
            "parent_noun": "nuclei of the parent cell type",
            "ref": "3month-protocol", "family": "all states of the same parent cell type"}


def column_guide(level):
    """(page, column, meaning) for every column of the three data pages."""
    w = level_words(level)
    unit_cols = ([("cell_type", "parent cell type"), ("cell_state", "cell state")] if level == "cell_state"
                 else [("cell_type", "cell type")])
    blank = "; blank if the heart has no nuclei of the parent cell type" if level == "cell_state" else ""
    rows = [("Per condition", c, m) for c, m in unit_cols] + [
        ("Per condition", "condition", "study arm"),
        ("Per condition", "n_samples", "hearts of the arm that enter the bar (= dots in the figure)"),
        ("Per condition", "abundance", f"nuclei of the {w['unit']}, summed over the hearts of the arm"),
        ("Per condition", "abundance_parent", f"{w['parent_noun']}, summed over the hearts of the arm"),
        ("Per condition", "proportion", "mean over the hearts of the arm of the per-heart % (= bar height in the figures)"),
        ("Per condition", "proportion_sem", "standard error of the per-heart % (= error bar); blank if < 2 hearts"),
        ("Per condition", "proportion_median", "median of the per-heart %"),
    ] + [("Per sample", c, m) for c, m in unit_cols] + [
        ("Per sample", "condition", "study arm"),
        ("Per sample", "Sample_ID", "heart (one heart = one animal = the unit of analysis)"),
        ("Per sample", "abundance", f"nuclei of the {w['unit']} in the heart"),
        ("Per sample", "abundance_parent", f"{w['parent_noun']} in the heart"),
        ("Per sample", "proportion", f"abundance / abundance_parent x 100 = % of {w['parent']} (= one dot in the figure){blank}"),
    ] + [("Statistics", c, m) for c, m in unit_cols] + [
        ("Statistics", "ref_condition", "reference arm of the comparison"),
        ("Statistics", "comp_condition", "arm compared with the reference"),
        ("Statistics", "n_samples_ref / n_samples_comp", "hearts in the reference / compared arm (as n_samples)"),
        ("Statistics", "abundance_in_ref / abundance_in_comp", f"nuclei of the {w['unit']} in the reference / compared arm"),
        ("Statistics", "proportion_in_ref / proportion_in_comp", "mean per-heart % in the reference / compared arm (bar heights)"),
        ("Statistics", "EffectSize", "CLR: difference in mean centred log-ratio, compared minus reference (> 0 = higher in the compared arm)"),
        ("Statistics", "MeanPropRef", "CLR: mean proportion (fraction 0-1) of the reference arm on CLR's zero-adjusted proportions (CLR column MeanPropControl)"),
        ("Statistics", "fold_change", "CLR: mean proportion compared / reference on CLR's zero-adjusted proportions (can differ from proportion_in_comp / proportion_in_ref)"),
        ("Statistics", "log2FC", "CLR: log2(fold_change)"),
        ("Statistics", "PValue", "CLR: p-value of the linear model on the centred log-ratio values"),
        ("Statistics", "FDR", f"CLR: Benjamini-Hochberg FDR within one comparison across {w['family']}"),
        ("Statistics", "significance", "*** FDR < 0.001, ** < 0.01, * < 0.05, ns = not significant, blank = no CLR result"),
    ]
    return rows


def page_notes(level):
    """Title and explanatory lines at the top of each data page."""
    w = level_words(level)
    no_parent = (["A heart without nuclei of the parent cell type has a blank proportion and is not part of that "
                  "cell type's figure."] if level == "cell_state" else [])
    return {
        "Per condition": (f"Per condition: abundance and proportion of each {w['unit']} in each study arm",
                          [f"One row per {w['unit']} x study arm.",
                           f"proportion = mean over the hearts of the arm of the % of {w['parent']} "
                           "(bar height in the figures); proportion_sem = error bar.",
                           "Column meanings: see the Guide page."]),
        "Per sample": (f"Per sample: nuclei and percentage of each {w['unit']} in each heart",
                       [f"One row per {w['unit']} x heart: the values behind the dots in the figures."] + no_parent
                       + ["Column meanings: see the Guide page."]),
        "Statistics": (f"Statistics: composition of both arms and the CLR result, per comparison",
                       [f"One row per {w['unit']} x comparison (comp_condition vs ref_condition); reference: {w['ref']}.",
                        "n_samples_ref ... proportion_in_comp = composition of the two arms (this notebook); "
                        "EffectSize ... FDR = CLR results from analysis/v1/CLR/v1, unchanged.",
                        "significance (CLR FDR): *** < 0.001, ** < 0.01, * < 0.05, ns = not significant, "
                        "blank = no CLR result (an arm without usable hearts).",
                        "Column meanings: see the Guide page."]),
    }


def excel_value(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        assert np.isfinite(v) or np.isnan(v), "infinite value cannot be written to Excel"
        return None if np.isnan(v) else float(v)
    return None if v == "" else v


def set_widths(ws, widths):
    for j, width in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(j)].width = width


def write_table(ws, top_row, df, n_id_cols):
    """Styled header at top_row, data below, number formats, filter, frozen header and id columns."""
    for j, col in enumerate(df.columns, start=1):
        c = ws.cell(row=top_row, column=j, value=col)
        c.font, c.fill, c.border = HEADER_FONT, HEADER_FILL, HEADER_BORDER
        c.alignment = Alignment(horizontal="center", vertical="center")
    for i, values in enumerate(df.itertuples(index=False), start=top_row + 1):
        for j, (col, v) in enumerate(zip(df.columns, values), start=1):
            c = ws.cell(row=i, column=j, value=excel_value(v))
            if col in NUMBER_FORMATS:
                c.number_format = NUMBER_FORMATS[col]
            if col == "significance":
                c.alignment = Alignment(horizontal="center")
    last = f"{get_column_letter(len(df.columns))}{top_row + len(df)}"
    ws.auto_filter.ref = f"A{top_row}:{last}"
    ws.freeze_panes = ws.cell(row=top_row + 1, column=n_id_cols + 1)
    set_widths(ws, [min(max(len(str(col)), 10) + 3, 40) if col not in (CELLSTATE_COL, CELLTYPE_COL, "condition", "comp_condition", "ref_condition")
                    else max(len(str(col)), int(df[col].astype(str).str.len().max())) + 3 for col in df.columns])


def write_guide(ws, level, csv_names):
    w = level_words(level)
    arms = ", ".join(f"{c} {int(design.loc[c, 'hearts'])}" for c in CONDITION_ORDER)
    ws.title = "Guide"
    ws.cell(row=1, column=1, value=f"{w['unit'].capitalize()} composition, preclinical snRNA-seq "
                                    "(pig-to-baboon cardiac xenotransplantation)").font = TITLE_FONT
    about = [
        f"Abundance (number of nuclei) and proportion of every {w['unit']} per heart and per study arm, with the "
        f"CLR statistics; the tables behind the {w['unit']} composition figures.",
        f"Proportion = % of {w['parent']}, averaged over the hearts of an arm (bar height in the figures).",
        f"Unit of analysis: one heart (Sample_ID). Hearts per arm: {arms}.",
        f"Statistics: CLR (centred log-ratio) linear model from analysis/v1/CLR/v1; reference arm {w['ref']}.",
        f"Generated by analysis/v1/composition/composition.ipynb on {date.today().isoformat()}.",
    ]
    for i, line in enumerate(about, start=3):
        ws.cell(row=i, column=1, value=line)
    r = len(about) + 4
    ws.cell(row=r, column=1, value="Pages").font = SECTION_FONT
    pages = pd.DataFrame({"Page": DATA_PAGES,
                          "Same table as CSV (tables/)": csv_names,
                          "What it shows": [f"abundance and proportion of each {w['unit']} in each arm",
                                            f"nuclei and % of each {w['unit']} in each heart (the dots)",
                                            "composition of both arms + CLR result, per comparison"]})
    write_table(ws, r + 1, pages, 1)
    ws.auto_filter.ref = None
    ws.freeze_panes = None
    r = r + 1 + len(pages) + 2
    ws.cell(row=r, column=1, value="Column guide").font = SECTION_FONT
    guide = pd.DataFrame(column_guide(level), columns=["Page", "Column", "Meaning"])
    write_table(ws, r + 1, guide, 1)
    ws.auto_filter.ref = None
    ws.freeze_panes = None
    set_widths(ws, [16, 56, 140])


def write_workbook(level, tables):
    """tables: {page name: (DataFrame, CSV file name)}"""
    wb = Workbook()
    write_guide(wb.active, level, [tables[p][1] for p in DATA_PAGES])
    notes = page_notes(level)
    n_id = 2 if level == "cell_state" else 1
    for page in DATA_PAGES:
        df = tables[page][0]
        ws = wb.create_sheet(page)
        title, lines = notes[page]
        ws.cell(row=1, column=1, value=title).font = TITLE_FONT
        for i, line in enumerate(lines, start=2):
            ws.cell(row=i, column=1, value=line).font = NOTE_FONT
        write_table(ws, len(lines) + 3, df, n_id)
    path = os.path.join(TABLE_DIR, f"{level}_composition_{FILE_TAG}.xlsx")
    wb.save(path)
    print(f"  saved workbook -> {os.path.relpath(path, OUT_DIR)}  (pages: Guide, {', '.join(DATA_PAGES)})")


for level, (pc, ps, st) in [("cell_type", (ct_per_condition, ct_per_sample, ct_stats)),
                            ("cell_state", (cs_per_condition, cs_per_sample, cs_stats))]:
    write_workbook(level, {
        "Per condition": (pc, f"{level}_composition_per_condition_{FILE_TAG}.csv"),
        "Per sample": (ps, f"{level}_composition_per_sample_{FILE_TAG}.csv"),
        "Statistics": (st, f"{level}_composition_stats_{FILE_TAG}.csv"),
    })

  saved workbook -> tables/cell_type_composition_preclinical.xlsx  (pages: Guide, Per condition, Per sample, Statistics)
  saved workbook -> tables/cell_state_composition_preclinical.xlsx  (pages: Guide, Per condition, Per sample, Statistics)


## 11. Figure function


In [11]:
# =============================================================================
# 11. FIGURE FUNCTION
# Every number drawn comes from the three tables passed in (per heart, per arm, statistics).
# =============================================================================

def plot_composition(per_sample, per_condition, stats, unit_col, unit_order, conditions, title, out_base):
    n_cond = len(conditions)
    if n_cond not in FIGSIZE_BY_NCOND:
        raise ValueError(f"No FIGSIZE_BY_NCOND entry for {n_cond} arms.")
    subplot_w, subplot_h = FIGSIZE_BY_NCOND[n_cond]
    x_pos = np.arange(n_cond) * BAR_SPACING

    n_unit = len(unit_order)
    n_cols = min(N_PANEL_COLS, n_unit)
    n_rows = int(np.ceil(n_unit / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(subplot_w * n_cols, subplot_h * n_rows), squeeze=False)
    axes_flat = axes.flatten()
    summary = per_condition.set_index([unit_col, "condition"])
    rng = np.random.default_rng(JITTER_SEED)

    for idx, unit in enumerate(unit_order):
        ax = axes_flat[idx]
        unit_data = per_sample[per_sample[unit_col] == unit]

        means = [summary.loc[(unit, c), "proportion"] for c in conditions]
        sems = [summary.loc[(unit, c), "proportion_sem"] for c in conditions]
        means = [0.0 if pd.isna(m) else m for m in means]
        sems = [0.0 if pd.isna(s) else s for s in sems]
        colors = [COND_COLORS[c] for c in conditions]

        # --- Bars (mean) and error bars (SEM) ---
        ax.bar(x_pos, means, width=BAR_WIDTH, color=colors, edgecolor="black",
               linewidth=1.5, alpha=0.75, zorder=2)
        ax.errorbar(x_pos, means, yerr=sems, fmt="none", ecolor="black", elinewidth=ERRORBAR_LW,
                    capsize=ERRORBAR_CAPSIZE, capthick=ERRORBAR_LW, zorder=3)

        # --- One dot per heart ---
        for x, cond in zip(x_pos, conditions):
            vals = unit_data.loc[unit_data["condition"] == cond, "proportion"].dropna().values
            if len(vals) == 0:
                continue
            jitter = rng.uniform(-POINT_JITTER, POINT_JITTER, size=len(vals))
            ax.scatter(np.full(len(vals), x) + jitter, vals, s=POINT_SIZE, alpha=POINT_ALPHA,
                       facecolors=COND_COLORS[cond], edgecolors="black", linewidths=1.5, zorder=4)

        # --- Axes styling ---
        ax.set_xlim(x_pos[0] - INNER_XPAD, x_pos[-1] + INNER_XPAD)
        ax.grid(False)
        ax.set_title(unit, fontsize=12, fontweight="bold", pad=8)
        ax.set_xlabel("")
        ax.set_ylabel("Proportion (%)", fontsize=10, fontweight="bold")
        ax.set_xticks(x_pos)
        ax.set_xticklabels(conditions, rotation=45, ha="right")
        ax.tick_params(axis="x", which="both", bottom=True, top=False, labelbottom=True,
                       labelsize=8, width=1.5, length=4, direction="out")
        ax.tick_params(axis="y", which="both", left=True, right=False, labelleft=True,
                       labelsize=8, width=1.5, length=4, direction="out")
        for tick in ax.get_xticklabels():
            tick.set_fontweight("bold")
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(1.8)
            ax.spines[side].set_color("black")

        # --- y-range: highest dot or bar + SEM ---
        all_vals = unit_data["proportion"].dropna().values
        bar_tops = [m + s for m, s in zip(means, sems)]
        max_val = max(np.max(all_vals) if len(all_vals) > 0 else 0.0, max(bar_tops) if bar_tops else 0.0)
        if pd.isna(max_val) or max_val == 0:
            max_val = 1.0

        # --- Significance brackets (CLR FDR < 0.05), read from the statistics table ---
        sig = stats[(stats[unit_col] == unit) & stats["significance"].isin(STAR_MARKS)]
        sig = sig.sort_values("FDR", kind="mergesort")
        if sig.empty:
            ax.set_ylim(0, max_val * 1.15)
            continue

        data_range = max_val
        base_y = max_val + Y_START_ABOVE_DATA * data_range
        bracket_h = BRACKET_H_FRAC * data_range
        star_offset = STAR_OFFSET_FRAC * data_range
        top_y = base_y + (len(sig) - 1) * Y_STEP * data_range + bracket_h + star_offset
        ax.set_ylim(0, top_y * 1.08)

        for layer, (_, row) in enumerate(sig.iterrows()):
            y_bar = base_y + layer * Y_STEP * data_range
            x1 = x_pos[conditions.index(row["ref_condition"])]
            x2 = x_pos[conditions.index(row["comp_condition"])]
            x1, x2 = min(x1, x2), max(x1, x2)
            ax.plot([x1, x1, x2, x2], [y_bar, y_bar + bracket_h, y_bar + bracket_h, y_bar],
                    color="black", lw=1.0, clip_on=False)
            ax.text((x1 + x2) / 2, y_bar + bracket_h + star_offset, row["significance"],
                    ha="center", va="bottom", fontsize=8, fontweight="bold", color="black", clip_on=False)

    for j in range(n_unit, len(axes_flat)):
        axes_flat[j].axis("off")

    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.0)
    handles = [mpatches.Patch(facecolor=COND_COLORS[c], edgecolor="black", linewidth=1.5, label=c, alpha=0.75)
               for c in conditions]
    fig.legend(handles=handles, title="Condition", title_fontsize=10, fontsize=10, loc="upper center",
               bbox_to_anchor=(0.5, -0.02), ncol=min(6, n_cond), frameon=True, framealpha=0.85,
               edgecolor="black")
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])

    os.makedirs(os.path.dirname(out_base), exist_ok=True)
    plt.savefig(f"{out_base}.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved figure -> {os.path.relpath(out_base, OUT_DIR)}.pdf / .png")


def draw_figure_set(folder):
    """Draw every figure of one set (configured in FIGURE_SETS) and list its significant comparisons."""
    level, ref, conditions = FIGURE_SETS[folder]
    print(f"=== {folder}  (reference {ref}; arms {conditions})")
    for parent in ([None] if level == "cell_type" else CELL_STATE_PARENTS):
        ps, pc, st = figure_tables(level, ref, conditions, parent)
        if level == "cell_type":
            unit_col, unit_order, title = CELLTYPE_COL, CELL_TYPE_ORDER, "Global Cell Type Compositional Analysis"
        else:
            unit_col, unit_order, title = CELLSTATE_COL, CELL_STATE_ORDER[parent], f"{parent} states compositional analysis"
        plot_composition(ps, pc, st, unit_col, unit_order, conditions, title,
                         figure_paths(folder, level, ref, parent)["figure"])
        sig = st[st["significance"].isin(STAR_MARKS)]
        if not sig.empty:
            print(sig[[unit_col, "comp_condition", "proportion_in_ref", "proportion_in_comp", "log2FC",
                       "PValue", "FDR", "significance"]]
                  .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## 12. Figures: cell types, reference 3month-protocol

In [12]:
draw_figure_set("cell_type_all/ref_3month")

=== cell_type_all/ref_3month  (reference 3month-protocol; arms ['Control', '3month-protocol', '6month-protocol', 'Prematurely-terminated', 'PCMV/PRV', 'Rejection'])


  saved figure -> cell_type_all/ref_3month/global_cell_types_composition_ref_3month_preclinical.pdf / .png
  cell_type         comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
  Lymph_sus                Control                  0              0.2424   9.417 5.463e-12 5.463e-11          ***
  Mye_papio                Control              4.357                   0  -8.916 7.245e-09 3.622e-08          ***
Lymph_papio                Control               1.79                   0  -7.633 2.415e-07 8.051e-07          ***
    Mye_sus                Control            0.02762               1.557   5.702 2.508e-05  6.27e-05          ***
         NC                Control             0.3181              0.6982   1.134  0.004146  0.008293           **
  Lymph_sus Prematurely-terminated                  0               0.183   9.012 1.098e-09 1.098e-08          ***
  Mye_papio Prematurely-terminated              4.357              0.2615  -4.058 1.132e

## 13. Figures: cell types, reference Control

In [13]:
draw_figure_set("cell_type_all/ref_control")

=== cell_type_all/ref_control  (reference Control; arms ['Control', '3month-protocol', '6month-protocol', 'Prematurely-terminated', 'PCMV/PRV', 'Rejection'])


  saved figure -> cell_type_all/ref_control/global_cell_types_composition_ref_control_preclinical.pdf / .png
  cell_type         comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
  Lymph_sus        3month-protocol             0.2424                   0  -9.417 5.463e-12 5.463e-11          ***
  Mye_papio        3month-protocol                  0               4.357   8.916 7.245e-09 3.622e-08          ***
Lymph_papio        3month-protocol                  0                1.79   7.633 2.415e-07 8.051e-07          ***
    Mye_sus        3month-protocol              1.557             0.02762  -5.702 2.508e-05  6.27e-05          ***
         NC        3month-protocol             0.6982              0.3181  -1.134  0.004146  0.008293           **
  Mye_papio        6month-protocol                  0               7.711    9.74 4.883e-07 4.883e-06          ***
    Mye_sus        6month-protocol              1.557             0.02425  -5.701 1.40

## 14. Figures: cell states, all arms

In [14]:
draw_figure_set("cell_state_all")

=== cell_state_all  (reference 3month-protocol; arms ['Control', '3month-protocol', '6month-protocol', 'Prematurely-terminated', 'PCMV/PRV', 'Rejection'])


  saved figure -> cell_state_all/CM/CM_states_composition_preclinical.pdf / .png
cell_state  comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
       CM2 6month-protocol               17.9               10.51 -0.7688 0.0003598  0.002518           **
       CM1 6month-protocol              38.46               27.26 -0.4967   0.01367   0.03189            *
       CM4 6month-protocol              10.91               8.217 -0.4088   0.01105   0.03189            *
       CM3 6month-protocol              13.58               10.96 -0.3094   0.01842   0.03224            *
Hypoxia_CM 6month-protocol              18.22               34.85  0.9353   0.03282   0.04595            *
    ISG_CM        PCMV/PRV              0.861               13.37   3.952  2.22e-05 0.0001554          ***
       CM1        PCMV/PRV              38.46               22.03 -0.8037  0.001178  0.004123           **
       CM4        PCMV/PRV              10.91               8.3

  saved figure -> cell_state_all/EC/EC_states_composition_preclinical.pdf / .png
 cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue      FDR significance
 ISG_cap_EC        Control              9.413               1.046  -3.029  0.002114  0.01903            *
     Cap_EC        Control              27.36               50.96  0.8894  0.007658  0.03446            *
PCMV/PRV_EC       PCMV/PRV                  0               5.772   6.311 5.633e-07 5.07e-06          ***
  Prolif_EC       PCMV/PRV              4.486             0.09704  -3.604  0.006112  0.02751            *
    Meso_EC       PCMV/PRV              1.748              0.3948  -1.437   0.01371  0.04113            *


  saved figure -> cell_state_all/FB/FB_states_composition_preclinical.pdf / .png
   cell_state         comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
     Angio_FB                Control              15.16              0.2531  -5.538 7.188e-05  0.000575          ***
     Homeo_FB                Control              51.76               71.69  0.4667 0.0004618  0.001231           **
Progenitor_FB                Control              5.151               8.657  0.7442 0.0003582  0.001231           **
       ECM_FB                Control               12.3               15.06  0.2909   0.01082   0.02163            *
     Homeo_FB        6month-protocol              51.76               24.77  -1.062 0.0005066  0.004053           **
     Angio_FB        6month-protocol              15.16               6.891   -1.13   0.01132    0.0453            *
Progenitor_FB Prematurely-terminated              5.151                14.2   1.455  0.002842   0.02

  saved figure -> cell_state_all/Lymph_papio/Lymph_papio_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_all/Lymph_sus/Lymph_sus_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_all/Mye_papio/Mye_papio_states_composition_preclinical.pdf / .png
          cell_state         comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
      LYVE1_MP_papio        6month-protocol              35.01               23.78 -0.5559  0.002652   0.02387            *
         AP_MP_papio Prematurely-terminated              17.32              0.8772  -2.858  0.002058  0.006174           **
Groundstate_MP_papio Prematurely-terminated              15.23               3.346  -1.694  0.001747  0.006174           **
            MO_papio Prematurely-terminated              4.416               32.49   2.657  0.001936  0.006174           **
      LYVE1_MP_papio Prematurely-terminated              35.01                8.09  -1.922  0.003711   0.00668           **
       SPP1_MP_papio Prematurely-terminated                7.8               26.82   1.662  0.003407   0.00668           **
        ISG_MP_papio Prematurely-term

  saved figure -> cell_state_all/Mye_sus/Mye_sus_states_composition_preclinical.pdf / .png
       cell_state         comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
        AP_MP_sus                Control                  0                20.5   2.265 0.0002417  0.002417           **
     LYVE1_MP_sus        6month-protocol              88.33                   0  -4.019  0.004382   0.04382            *
Phagocytic_MP_sus Prematurely-terminated                  0               29.53   2.804 4.038e-05 0.0004038          ***
           DC_sus Prematurely-terminated                  0               9.064   1.104  0.005095   0.01689            *
       ISG_MP_sus Prematurely-terminated                  0              0.5262  -2.911  0.003396   0.01689            *
         Mast_sus Prematurely-terminated                  0               1.161  -1.822  0.006757   0.01689            *
       ISG_MP_sus               PCMV/PRV                  0   

## 15. Figures: cell states, 6month vs 3month

In [15]:
draw_figure_set("cell_state_6m_3m")

=== cell_state_6m_3m  (reference 3month-protocol; arms ['3month-protocol', '6month-protocol'])


  saved figure -> cell_state_6m_3m/CM/CM_states_composition_preclinical.pdf / .png
cell_state  comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue      FDR significance
       CM2 6month-protocol               17.9               10.51 -0.7688 0.0003598 0.002518           **
       CM1 6month-protocol              38.46               27.26 -0.4967   0.01367  0.03189            *
       CM4 6month-protocol              10.91               8.217 -0.4088   0.01105  0.03189            *
       CM3 6month-protocol              13.58               10.96 -0.3094   0.01842  0.03224            *
Hypoxia_CM 6month-protocol              18.22               34.85  0.9353   0.03282  0.04595            *


  saved figure -> cell_state_6m_3m/EC/EC_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_6m_3m/FB/FB_states_composition_preclinical.pdf / .png
cell_state  comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue      FDR significance
  Homeo_FB 6month-protocol              51.76               24.77  -1.062 0.0005066 0.004053           **
  Angio_FB 6month-protocol              15.16               6.891   -1.13   0.01132   0.0453            *


  saved figure -> cell_state_6m_3m/Lymph_papio/Lymph_papio_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_6m_3m/Lymph_sus/Lymph_sus_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_6m_3m/Mye_papio/Mye_papio_states_composition_preclinical.pdf / .png
    cell_state  comp_condition  proportion_in_ref  proportion_in_comp  log2FC   PValue     FDR significance
LYVE1_MP_papio 6month-protocol              35.01               23.78 -0.5559 0.002652 0.02387            *


  saved figure -> cell_state_6m_3m/Mye_sus/Mye_sus_states_composition_preclinical.pdf / .png
  cell_state  comp_condition  proportion_in_ref  proportion_in_comp  log2FC   PValue     FDR significance
LYVE1_MP_sus 6month-protocol              88.33                   0  -4.019 0.004382 0.04382            *


## 16. Figures: cell states, 3month, PCMV/PRV, Rejection

In [16]:
draw_figure_set("cell_state_3month_PCMV_Rej")

=== cell_state_3month_PCMV_Rej  (reference 3month-protocol; arms ['3month-protocol', 'PCMV/PRV', 'Rejection'])


  saved figure -> cell_state_3month_PCMV_Rej/CM/CM_states_composition_preclinical.pdf / .png
cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC   PValue       FDR significance
    ISG_CM       PCMV/PRV              0.861               13.37   3.952 2.22e-05 0.0001554          ***
       CM1       PCMV/PRV              38.46               22.03 -0.8037 0.001178  0.004123           **
       CM4       PCMV/PRV              10.91               8.356 -0.3845  0.01114     0.026            *
       CM2       PCMV/PRV               17.9               15.61 -0.1979  0.01966   0.03441            *
       CM1      Rejection              38.46               28.67 -0.4236  0.00503   0.03521            *
       CM3      Rejection              13.58                11.8 -0.2033  0.01114     0.039            *


  saved figure -> cell_state_3month_PCMV_Rej/EC/EC_states_composition_preclinical.pdf / .png
 cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue      FDR significance
PCMV/PRV_EC       PCMV/PRV                  0               5.772   6.311 5.633e-07 5.07e-06          ***
  Prolif_EC       PCMV/PRV              4.486             0.09704  -3.604  0.006112  0.02751            *
    Meso_EC       PCMV/PRV              1.748              0.3948  -1.437   0.01371  0.04113            *


  saved figure -> cell_state_3month_PCMV_Rej/FB/FB_states_composition_preclinical.pdf / .png
   cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
     Homeo_FB      Rejection              51.76               24.23  -1.093 6.255e-05 0.0005004          ***
Progenitor_FB      Rejection              5.151               3.014 -0.7694  0.004316   0.01726            *


  saved figure -> cell_state_3month_PCMV_Rej/Lymph_papio/Lymph_papio_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_3month_PCMV_Rej/Lymph_sus/Lymph_sus_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_3month_PCMV_Rej/Mye_papio/Mye_papio_states_composition_preclinical.pdf / .png
          cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
      LYVE1_MP_papio       PCMV/PRV              35.01              0.4016  -3.528 2.123e-06 1.911e-05          ***
       SPP1_MP_papio       PCMV/PRV                7.8               21.26   1.573  0.002504   0.01127            *
         AP_MP_papio       PCMV/PRV              17.32               1.205  -2.183  0.005109   0.01533            *
        ISG_MP_papio       PCMV/PRV              8.908               23.45   1.497   0.01113   0.02505            *
       GBE1_MP_papio      Rejection              3.819                17.5   2.182   0.01096   0.03289            *
Groundstate_MP_papio      Rejection              15.23               11.74 -0.3732   0.01037   0.03289            *
      LYVE1_MP_papio      Rejection              35.01               16.12  -1.11

  saved figure -> cell_state_3month_PCMV_Rej/Mye_sus/Mye_sus_states_composition_preclinical.pdf / .png
       cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue      FDR significance
       ISG_MP_sus       PCMV/PRV                  0               38.23   3.082 0.0003589 0.003589           **
Phagocytic_MP_sus       PCMV/PRV                  0               9.105   1.119   0.00137  0.00685           **
      VCAN_MO_sus       PCMV/PRV                  0              0.7348  -1.651  0.002865 0.009551           **
     LYVE1_MP_sus       PCMV/PRV              88.33               11.96  -2.266  0.008493  0.02123            *
      SPP1_MP_sus       PCMV/PRV                  0               17.65   2.056   0.01152  0.02303            *
         Mast_sus       PCMV/PRV                  0               1.107  -1.301   0.01689  0.02815            *
         Mast_sus      Rejection                  0              0.1458  -3.581  0.001093  0.01093            *
 

## 17. Figures — cell states, Rejection vs 3month

In [17]:
draw_figure_set("cell_state_3month_Rej")

=== cell_state_3month_Rej  (reference 3month-protocol; arms ['3month-protocol', 'Rejection'])


  saved figure -> cell_state_3month_Rej/CM/CM_states_composition_preclinical.pdf / .png
cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC  PValue     FDR significance
       CM1      Rejection              38.46               28.67 -0.4236 0.00503 0.03521            *
       CM3      Rejection              13.58                11.8 -0.2033 0.01114   0.039            *


  saved figure -> cell_state_3month_Rej/EC/EC_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_3month_Rej/FB/FB_states_composition_preclinical.pdf / .png
   cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC    PValue       FDR significance
     Homeo_FB      Rejection              51.76               24.23  -1.093 6.255e-05 0.0005004          ***
Progenitor_FB      Rejection              5.151               3.014 -0.7694  0.004316   0.01726            *


  saved figure -> cell_state_3month_Rej/Lymph_papio/Lymph_papio_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_3month_Rej/Lymph_sus/Lymph_sus_states_composition_preclinical.pdf / .png


  saved figure -> cell_state_3month_Rej/Mye_papio/Mye_papio_states_composition_preclinical.pdf / .png
          cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC   PValue     FDR significance
       GBE1_MP_papio      Rejection              3.819                17.5   2.182  0.01096 0.03289            *
Groundstate_MP_papio      Rejection              15.23               11.74 -0.3732  0.01037 0.03289            *
      LYVE1_MP_papio      Rejection              35.01               16.12  -1.115 0.003829 0.03289            *


  saved figure -> cell_state_3month_Rej/Mye_sus/Mye_sus_states_composition_preclinical.pdf / .png
 cell_state comp_condition  proportion_in_ref  proportion_in_comp  log2FC   PValue     FDR significance
   Mast_sus      Rejection                  0              0.1458  -3.581 0.001093 0.01093            *
SPP1_MP_sus      Rejection                  0                8.66   1.045  0.00229 0.01145            *


## 18. Consistency checks and significant results

In [18]:

# a. counts add up
arm_nuclei = obs.groupby(CONDITION_COL, observed=True).size()
arm_nuclei.index = arm_nuclei.index.astype(str)
ct_sum = ct_per_condition.groupby("condition")["abundance"].sum()
assert (ct_sum.reindex(arm_nuclei.index) == arm_nuclei).all()
cs_sum = cs_per_condition.groupby([CELLTYPE_COL, "condition"])["abundance"].sum()
ct_abund = ct_per_condition.set_index([CELLTYPE_COL, "condition"])["abundance"]
assert (cs_sum == ct_abund.reindex(cs_sum.index)).all()
print("a. abundances add up to the nuclei totals: OK")

# b. statistics CSVs on disk == master tables == CLR source values
clr_all = {CELLTYPE_COL: clr_cell_type, CELLSTATE_COL: clr_cell_state}
master = {CELLTYPE_COL: ct_stats, CELLSTATE_COL: cs_stats}
stat_files = sorted(glob.glob(os.path.join(TABLE_DIR, "*_composition_stats_*.csv"))
                    + glob.glob(os.path.join(OUT_DIR, "cell_*", "**", "*_composition_*.csv"), recursive=True))
for path in stat_files:
    saved = pd.read_csv(path, keep_default_na=False, na_values=[""])
    level = CELLSTATE_COL if CELLSTATE_COL in saved.columns else CELLTYPE_COL
    key = [CELLTYPE_COL] + ([CELLSTATE_COL] if level == CELLSTATE_COL else []) + ["ref_condition", "comp_condition"]
    m = saved.merge(master[level], on=key, suffixes=("", "_master"), validate="one_to_one")
    assert len(m) == len(saved), path
    for col in ["proportion_in_ref", "proportion_in_comp"] + CLR_VALUE_COLS:
        assert np.allclose(m[col], m[f"{col}_master"], rtol=1e-5, equal_nan=True), (path, col)
    c = saved.merge(clr_all[level], on=key, suffixes=("", "_clr"), how="inner")
    assert len(c) == saved["PValue"].notna().sum(), path
    for col in CLR_VALUE_COLS:
        assert np.allclose(c[col], c[f"{col}_clr"], rtol=1e-5), (path, col)
    assert (m["significance"].fillna("") == m["significance_master"]).all(), path
print(f"b. {len(stat_files)} statistics CSVs match the master tables and the CLR files: OK")

# c. stars follow the CLR FDR
for st in (ct_stats, cs_stats):
    assert (st["FDR"].map(significance_stars) == st["significance"]).all()
print("c. stars follow the CLR FDR thresholds: OK")

# d. every figure set is complete on disk
n_expected = 0
for folder, (level, ref, conditions) in FIGURE_SETS.items():
    for parent in ([None] if level == "cell_type" else CELL_STATE_PARENTS):
        p = figure_paths(folder, level, ref, parent)
        for f in [f"{p['figure']}.pdf", f"{p['figure']}.png", p["stats"], p["per_condition"]]:
            if f is not None:
                assert os.path.exists(f), f
                n_expected += 1
ct_pc_file = os.path.join(OUT_DIR, "cell_type_all", f"global_cell_types_abundance_per_condition_{FILE_TAG}.csv")
assert os.path.exists(ct_pc_file), ct_pc_file
print(f"d. {n_expected + 1} figure-set files present: OK")

# e. every Excel data page equals its CSV
for level in ("cell_type", "cell_state"):
    xlsx = os.path.join(TABLE_DIR, f"{level}_composition_{FILE_TAG}.xlsx")
    pages = pd.read_excel(xlsx, sheet_name=None, header=None)
    assert list(pages) == ["Guide"] + DATA_PAGES, list(pages)
    for page, kind in zip(DATA_PAGES, ["per_condition", "per_sample", "stats"]):
        csv = pd.read_csv(os.path.join(TABLE_DIR, f"{level}_composition_{kind}_{FILE_TAG}.csv"),
                          keep_default_na=False, na_values=[""])
        raw = pages[page]
        top = raw.index[(raw[0] == csv.columns[0]) & (raw[1] == csv.columns[1])][0]
        sheet = raw.iloc[top + 1:].reset_index(drop=True)
        sheet.columns = list(raw.iloc[top])
        assert list(sheet.columns) == list(csv.columns) and len(sheet) == len(csv), (xlsx, page)
        for col in csv.columns:
            if pd.api.types.is_numeric_dtype(csv[col]):
                assert np.allclose(pd.to_numeric(sheet[col]), csv[col], rtol=1e-5, equal_nan=True), (xlsx, page, col)
            else:
                assert (sheet[col].fillna("").astype(str) == csv[col].fillna("").astype(str)).all(), (xlsx, page, col)
print("e. every Excel data page equals its CSV: OK\n")

show = ["ref_condition", "comp_condition", "n_samples_ref", "n_samples_comp", "proportion_in_ref",
        "proportion_in_comp", "EffectSize", "log2FC", "PValue", "FDR", "significance"]
for name, st, unit in [("CELL TYPES", ct_stats, [CELLTYPE_COL]),
                       ("CELL STATES", cs_stats, [CELLTYPE_COL, CELLSTATE_COL])]:
    sig = st[st["significance"].isin(STAR_MARKS)]
    print(f"=== {name}: {len(sig)} of {st['PValue'].notna().sum()} CLR tests with FDR < 0.05 ===")
    print(sig[unit + show].to_string(index=False, float_format=lambda v: f"{v:.4g}"), "\n")

a. abundances add up to the nuclei totals: OK
b. 32 statistics CSVs match the master tables and the CLR files: OK
c. stars follow the CLR FDR thresholds: OK
d. 119 figure-set files present: OK


e. every Excel data page equals its CSV: OK

=== CELL TYPES: 35 of 100 CLR tests with FDR < 0.05 ===
  cell_type   ref_condition         comp_condition  n_samples_ref  n_samples_comp  proportion_in_ref  proportion_in_comp  EffectSize  log2FC    PValue       FDR significance
  Lymph_sus 3month-protocol                Control              7               4                  0              0.2424       6.648   9.417 5.463e-12 5.463e-11          ***
  Mye_papio 3month-protocol                Control              7               4              4.357                   0      -5.957  -8.916 7.245e-09 3.622e-08          ***
Lymph_papio 3month-protocol                Control              7               4               1.79                   0      -4.872  -7.633 2.415e-07 8.051e-07          ***
    Mye_sus 3month-protocol                Control              7               4            0.02762               1.557       4.448   5.702 2.508e-05  6.27e-05          ***
         NC 3month-protocol  